# GEO project

Which headphone brands do AI assistants recommend?

I asked Gemini 30 shopping questions. 26 of them have no brand names, and those are used for the results (the other 4
compare two brands). Then I counted which brands appeared in the answers and which one came first. I also checked Sennheiser's product page to see how readable it is for AI.

**Tools:** Python, Google Colab, Gemini, DuckDB + SQL, pandas, Plotly, BeautifulSoup/Playwright, Google Sheets

**How to run:** Runtime → Run all (about 10 minutes). The results are saved to Google Drive in the `geo-project` folder.

## Setup

In [1]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
INSTALL = IN_COLAB
INSTALL_BROWSER = True     # Playwright Chromium for the page check (JavaScript rendering); False = raw HTML only

if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "anthropic", "duckdb", "plotly",
                    "httpx", "beautifulsoup4", "lxml", "playwright", "gspread", "markdown"], check=True)
    if INSTALL_BROWSER:
        r = subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"],
                           capture_output=True, text=True)
        print("Chromium:", "ready" if r.returncode == 0 else "not installed (raw HTML only)")

Chromium: ready


In [2]:
import re, json, math, time, random, hashlib, threading, html as htmllib
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse
from urllib import robotparser
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import duckdb
import httpx
from bs4 import BeautifulSoup
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

pd.set_option("display.max_colwidth", 120)

## Settings
The model, brands and questions.

In [3]:
PROVIDER = "colab_ai"        # "colab_ai" (free, no key) | "gemini" (free AI Studio key; web search only if your tier allows) | "anthropic" (paid)
PROVIDER = os.environ.get("GEO_PROJECT_PROVIDER", PROVIDER)
DRY_RUN = False              # True = simulated responses (offline test, labelled SIMULATED)
DRY_RUN = DRY_RUN or os.environ.get("GEO_PROJECT_DRY_RUN") == "1"
REPEATS = 1                  # times each question is asked (2+ gives tighter confidence intervals)
USE_LLM_EXTRACTION = True    # False halves the number of model calls (brands are still detected in SQL)
EXPORT_TO_SHEETS = True      # write the marts to a Google Sheet for Looker Studio (Colab only)

MODEL_MAIN = "auto"          # answers the shopper questions
MODEL_FAST = "auto"          # structured extraction and the memo
DEFAULT_MODELS = {"anthropic": ("claude-sonnet-5-5", "claude-haiku-5-5")}
RPM_LIMIT = {"anthropic": None, "gemini": 8, "colab_ai": 15}[PROVIDER]
MAX_WORKERS = 4 if PROVIDER == "anthropic" else 2
USE_WEB_SEARCH = True
WEB_SEARCH_TOOL = {"type": "web_search_20250305", "name": "web_search", "max_uses": 3}
PRICES_PER_MTOK = {}

# ---------- Market ----------
CATEGORY = "noise-cancelling headphones"
MARKET_SUFFIX = " I'm based in the UK."
FOCUS_BRAND = "Sennheiser"
FOCUS_URL = "https://uk.sennheiser-hearing.com/products/momentum-4-wireless"   # page checked in stage 5

BRANDS = {   # canonical brand -> aliases (case-sensitive, whole words)
    "Sony": ["Sony", "WH-1000XM", "WF-1000XM", "ULT Wear"],
    "Bose": ["Bose", "QuietComfort", "QC Ultra"],
    "Sennheiser": ["Sennheiser", "Momentum 4", "Accentum"],
    "Apple": ["AirPods Max", "AirPods Pro", "AirPods"],
    "Beats": ["Beats"],
    "JBL": ["JBL"],
    "Anker Soundcore": ["Soundcore", "Anker"],
    "Bowers & Wilkins": ["Bowers & Wilkins", "Bowers and Wilkins", "B&W", "Px7", "Px8"],
    "Jabra": ["Jabra"],
    "Audio-Technica": ["Audio-Technica", "Audio Technica"],
    "Marshall": ["Marshall"],
    "Bang & Olufsen": ["Bang & Olufsen", "Bang and Olufsen", "Beoplay"],
    "Samsung": ["Galaxy Buds", "Samsung"],
    "Nothing": ["Nothing Headphone", "Nothing Ear", "CMF Headphone"],
}

# ---------- Shopper questions by intent (comparison prompts name brands, so they are excluded from headline KPIs) ----------
PROMPTS = {
    "budget": [f"What are the best {CATEGORY} under £{p}?" for p in (100, 150, 200, 250, 300)],
    "use_case": [f"Which {CATEGORY} would you recommend for {u}?" for u in (
        "long-haul flights", "commuting on the London Underground", "working from home and video calls",
        "the gym and running", "people who wear glasses", "studying in a noisy library",
        "a noisy open-plan office", "people with small ears")],
    "attribute": [f"Which {CATEGORY} have the {a}?" for a in (
        "best sound quality", "longest battery life", "best noise cancellation", "best microphone for calls",
        "most comfortable fit for all-day wear", "best value for money", "lightest weight", "best app and EQ")],
    "generic": [f"What are the best {CATEGORY} right now?", f"I want to buy new {CATEGORY}. What should I get?",
                f"Recommend some {CATEGORY} for an iPhone user.", f"Recommend some {CATEGORY} for an Android user.",
                f"Which {CATEGORY} do audio experts recommend?"],
    "comparison": [f"{a} or {b}: which {CATEGORY} should I buy?" for a, b in (
        ("Sony", "Bose"), ("Sennheiser", "Sony"), ("Bose", "Sennheiser"), ("Apple AirPods Max", "Sony"))],
}

# ---------- Domain taxonomy for cited sources (suffix match) ----------
DOMAINS = {
    "Brand site": {"sony.co.uk": "Sony", "sony.com": "Sony", "bose.co.uk": "Bose", "bose.com": "Bose",
                   "sennheiser.com": "Sennheiser", "sennheiser-hearing.com": "Sennheiser", "apple.com": "Apple",
                   "beatsbydre.com": "Beats", "jbl.com": "JBL", "soundcore.com": "Anker Soundcore",
                   "bowerswilkins.com": "Bowers & Wilkins", "jabra.com": "Jabra", "jabra.co.uk": "Jabra",
                   "audio-technica.com": "Audio-Technica", "marshall.com": "Marshall",
                   "bang-olufsen.com": "Bang & Olufsen", "samsung.com": "Samsung", "nothing.tech": "Nothing"},
    "Retailer": ["amazon.co.uk", "amazon.com", "currys.co.uk", "argos.co.uk", "johnlewis.com", "richersounds.com",
                 "very.co.uk", "box.co.uk", "scan.co.uk", "ebay.co.uk", "selfridges.com", "petertyson.co.uk",
                 "sevenoakssoundandvision.co.uk", "hifigear.co.uk"],
    "Review / media": ["whathifi.com", "techradar.com", "rtings.com", "theverge.com", "tomsguide.com", "which.co.uk",
                       "trustedreviews.com", "soundguys.com", "t3.com", "stuff.tv", "expertreviews.co.uk",
                       "independent.co.uk", "theguardian.com", "wired.com", "wired.co.uk", "cnet.com", "engadget.com",
                       "pcmag.com", "telegraph.co.uk", "zdnet.com", "digitaltrends.com", "techadvisor.com",
                       "pocket-lint.com", "bbc.co.uk", "standard.co.uk", "gq-magazine.co.uk"],
    "Community / video": ["reddit.com", "youtube.com", "head-fi.org", "quora.com", "avforums.com", "tiktok.com"],
}

# ---------- Storage ----------
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/geo-project"
BASE_DIR = Path("geo_project")
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path(DRIVE_DIR)
CACHE_DIR = BASE_DIR / "cache" / ("dry_run" if DRY_RUN else PROVIDER)
SQL_DIR = BASE_DIR / "sql"
for d in (BASE_DIR, CACHE_DIR, SQL_DIR):
    d.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE_DIR / ("geo_project_SIMULATED.duckdb" if DRY_RUN else "geo_project.duckdb")
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
RUN_TS = time.strftime("%Y-%m-%d %H:%M:%S")
n_unbranded = sum(len(v) for k, v in PROMPTS.items() if k != "comparison")
print(f"Run {RUN_ID} | provider {PROVIDER} | {n_unbranded} unbranded + {len(PROMPTS['comparison'])} branded questions "
      f"x {REPEATS} | data: {BASE_DIR.resolve()}")

Mounted at /content/drive
Run 20261125-144739 | provider colab_ai | 26 unbranded + 4 branded questions x 1 | data: /content/drive/MyDrive/geo-project


In [4]:
def _secret(name):
    v = os.environ.get(name)
    if not v and IN_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
        except Exception:
            v = None
    return v

API_KEY = None
if not DRY_RUN and PROVIDER in ("gemini", "anthropic"):
    names = ["GEMINI_API_KEY", "GOOGLE_API_KEY"] if PROVIDER == "gemini" else ["ANTHROPIC_API_KEY"]
    API_KEY = next((v for v in map(_secret, names) if v), None)
    if not API_KEY:
        import getpass
        API_KEY = getpass.getpass(f"Paste your {names[0]} (or set PROVIDER = 'colab_ai'): ").strip()
print({"colab_ai": "No key needed"}.get(PROVIDER, "Key loaded" if API_KEY else "No key (dry run)"))

No key needed


## AI client
A helper that sends questions to the model. It saves answers (cache) and waits when there is a rate limit.

In [5]:
from concurrent.futures import TimeoutError as FuturesTimeout

FAKE_HANDLERS = {}   # purpose -> fn(prompt, kwargs, rng) -> dict ; each module registers its own


class QuotaExhausted(RuntimeError):
    pass


class ModelUnavailable(RuntimeError):
    pass


class RateLimiter:
    def __init__(self, rpm=None):
        self.interval = 60.0 / rpm if rpm else 0.0
        self.lock, self.next_t = threading.Lock(), 0.0

    def wait(self):
        if not self.interval:
            return
        with self.lock:
            now = time.time()
            t = max(now, self.next_t)
            self.next_t = t + self.interval
        time.sleep(max(0.0, t - now))


def _version_key(name):
    nums = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)", name.split("/")[-1])]
    return (nums[0] if nums else 0.0, "preview" not in name and "exp" not in name)


def pick_models(names, kind):
    bad = ("image", "tts", "audio", "live", "embedding", "native", "vision", "robotics", "computer",
           "learnlm", "gemma", "aqa", "thinking")
    cands = [n for n in names if "gemini" in n.lower() and "flash" in n.lower()
             and not any(b in n.lower() for b in bad)]
    lite = [n for n in cands if "lite" in n.lower()]
    full = [n for n in cands if "lite" not in n.lower()]
    ordered = (lite + full) if kind == "fast" else (full + lite)
    first = sorted(lite if kind == "fast" else full, key=_version_key, reverse=True)
    rest = sorted([n for n in ordered if n not in first], key=_version_key, reverse=True)
    return first + rest


class LLMClient:
    def __init__(self, provider, api_key=None, cache_dir=CACHE_DIR, dry_run=False, rpm=None):
        self.provider, self.dry_run = provider, dry_run
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.usage, self.lock = {}, threading.Lock()
        self.limiter = RateLimiter(rpm)
        self.web_search_ok = provider != "colab_ai"
        self.temperature_ok = provider != "colab_ai"
        self.exhausted = False
        self.consecutive_quota = 0
        self.max_attempts = 5
        self.max_wait = 60
        self.timeout_s = 120
        self.candidates, self.override = {"main": [], "fast": []}, {}
        self.client = None
        if dry_run:
            return
        if provider == "anthropic":
            import anthropic
            self._anthropic = anthropic
            self.client = anthropic.Anthropic(api_key=api_key, max_retries=6)
        elif provider == "gemini":
            from google import genai
            from google.genai import types
            self.types = types
            try:   # never wait forever on a stuck request
                self.client = genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=self.timeout_s * 1000))
            except Exception:
                self.client = genai.Client(api_key=api_key)
        elif provider == "colab_ai":
            try:
                from google.colab import ai
            except ImportError as e:
                raise RuntimeError("PROVIDER='colab_ai' only works inside Google Colab. "
                                   "Use PROVIDER='gemini' with a free AI Studio key instead.") from e
            self.client = ai
        else:
            raise ValueError(f"Unknown provider: {provider}")

    # ---------- models ----------
    def list_models(self):
        if self.provider == "gemini":
            out = []
            for m in self.client.models.list():
                acts = getattr(m, "supported_actions", None) or []
                if not acts or "generateContent" in acts:
                    out.append(m.name.replace("models/", ""))
            return out
        if self.provider == "colab_ai":
            return [str(m) for m in self.client.list_models()]
        return []

    def resolve_models(self, main="auto", fast="auto"):
        if self.dry_run:
            return "dry-run-main", "dry-run-fast"
        if self.provider == "anthropic":
            d = DEFAULT_MODELS["anthropic"]
            return (d[0] if main == "auto" else main), (d[1] if fast == "auto" else fast)
        names = self.list_models()
        self.candidates["main"] = [main] if main != "auto" else pick_models(names, "main")
        self.candidates["fast"] = [fast] if fast != "auto" else pick_models(names, "fast")
        if not self.candidates["main"]:
            raise RuntimeError(f"No suitable Gemini model found. Available: {names[:25]}. Set MODEL_MAIN manually.")
        if not self.candidates["fast"]:
            self.candidates["fast"] = self.candidates["main"]
        return self.candidates["main"][0], self.candidates["fast"][0]

    def _actual(self, model):
        seen = set()
        while model in self.override and model not in seen:
            seen.add(model)
            model = self.override[model]
        return model

    def _next_candidate(self, model):
        for lst in self.candidates.values():
            if model in lst:
                for m in lst[lst.index(model) + 1:]:
                    if m not in self.override:
                        return m
        return None

    # ---------- helpers ----------
    @staticmethod
    def _key(purpose, tag, kwargs):
        blob = json.dumps({"p": purpose, "t": tag, "k": kwargs}, sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:32]

    @staticmethod
    def _has_web_search(kwargs):
        return any(str(t.get("type", "")).startswith("web_search") for t in kwargs.get("tools", []))

    @staticmethod
    def _strip_web_search(kwargs):
        kwargs = dict(kwargs)
        tools = [t for t in kwargs.get("tools", []) if not str(t.get("type", "")).startswith("web_search")]
        if tools:
            kwargs["tools"] = tools
        else:
            kwargs.pop("tools", None)
        return kwargs

    @staticmethod
    def _forced_tool(kwargs):
        tc = kwargs.get("tool_choice") or {}
        if tc.get("type") == "tool":
            return next((t for t in kwargs.get("tools", []) if t.get("name") == tc.get("name")), None)
        return None

    @staticmethod
    def _parse_json(text, tool):
        t = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", (text or "").strip())
        obj = None
        try:
            obj = json.loads(t)
        except Exception:
            m = re.search(r"[\{\[].*[\}\]]", t, flags=re.S)
            if m:
                try:
                    obj = json.loads(m.group(0))
                except Exception:
                    obj = None
        if isinstance(obj, list):
            props = list(tool["input_schema"].get("properties", {}))
            obj = {props[0]: obj} if len(props) == 1 else {"items": obj}
        return obj

    def _prompt_parts(self, kwargs):
        system = kwargs.get("system") or ""
        user = kwargs["messages"][-1]["content"]
        tool = self._forced_tool(kwargs)
        if tool:
            user += ("\n\nReturn ONLY a JSON object (no prose, no code fences) that validates against this "
                     "JSON Schema:\n" + json.dumps(tool["input_schema"]))
        return system, user, tool

    def _retry(self, fn, max_attempts=None):
        max_attempts = max_attempts or self.max_attempts
        delay = 5.0
        last_quota = False
        for attempt in range(1, max_attempts + 1):
            self.limiter.wait()
            try:
                out = fn()
                with self.lock:
                    self.consecutive_quota = 0
                return out
            except Exception as e:
                msg = str(e) or type(e).__name__
                low = msg.lower().replace("_", "")
                if "404" in msg or "not found" in low or "limit: 0" in low:
                    raise ModelUnavailable(msg[:300])
                quota = "429" in msg or "resourceexhausted" in low or "quota" in low or "rate limit" in low
                if quota and ("perday" in low or "per day" in low or "daily" in low):
                    raise QuotaExhausted(msg[:400])
                transient = any(s in low for s in ("500", "502", "503", "504", "unavailable", "overloaded",
                                                   "deadline", "timeout", "timed out", "internal"))
                is_timeout = "timeout" in low or "timed out" in low
                if quota or transient:
                    last_quota = quota or is_timeout
                    if attempt == max_attempts or (is_timeout and attempt >= 2):
                        break
                    m = re.search(r"retry in ([\d.]+)s", low) or re.search(r"retrydelay\W+(\d+)", low)
                    wait = min(float(m.group(1)) + 1 if m else delay, self.max_wait)
                    print(f"  ... {'rate limit' if quota else 'temporary error'}: waiting {wait:.0f}s "
                          f"(attempt {attempt}/{max_attempts})", flush=True)
                    time.sleep(wait)
                    delay = min(delay * 2, 60)
                    continue
                raise
        if last_quota:
            with self.lock:
                self.consecutive_quota += 1
                n = self.consecutive_quota
            if n >= 3:
                raise QuotaExhausted("Repeated rate-limit or timeout errors: the free quota is probably used up, "
                                     "or the model is not responding. Try again later or switch PROVIDER.")
            raise RuntimeError("Rate limited or timed out after retries; skipping this request.")
        raise RuntimeError("Gave up after repeated temporary errors.")

    def _with_fallback(self, model, call):
        while True:
            try:
                return model, call(model)
            except ModelUnavailable as e:
                nxt = self._next_candidate(model)
                if not nxt:
                    raise QuotaExhausted(f"No available model left ({e})")
                print(f"Model '{model}' unavailable on this tier -> switching to '{nxt}'")
                self.override[model] = nxt
                model = nxt

    def probe(self, model, web_search=False, attempts=2):
        """Quick live check of one model/search combination. Returns (ok, short_error). Never cached."""
        if self.dry_run:
            return True, ""
        kw = dict(model=model, max_tokens=20, messages=[{"role": "user", "content": "Reply with one word: ok"}])
        if web_search:
            kw["tools"] = [{"type": "web_search_20250305", "name": "web_search", "max_uses": 1}]
        saved = (self.max_attempts, self.web_search_ok, dict(self.override), self.max_wait)
        self.max_attempts, self.max_wait = attempts, 8
        try:
            caller = {"anthropic": self._call_anthropic, "gemini": self._call_gemini, "colab_ai": self._call_colab}[self.provider]
            out = caller(kw)
            ok = bool(out.get("text")) and (not web_search or self.web_search_ok)
            return ok, "" if ok else "search not available"
        except Exception as e:
            return False, f"{type(e).__name__}: {str(e)[:120]}"
        finally:
            self.max_attempts, self.web_search_ok, self.override, self.max_wait = saved
            self.consecutive_quota, self.exhausted = 0, False

    def _track(self, purpose, model, usage, cached):
        with self.lock:
            u = self.usage.setdefault(purpose, {"model": self._actual(model) if model else model, "calls": 0,
                                                "cache_hits": 0, "input_tokens": 0, "output_tokens": 0,
                                                "web_searches": 0})
            u["calls"] += 1
            u["cache_hits"] += int(cached)
            if not cached:
                for k in ("input_tokens", "output_tokens", "web_searches"):
                    u[k] += usage.get(k, 0) or 0

    # ---------- public API ----------
    def create(self, purpose, tag="", **kwargs):
        """Anthropic-style arguments in; normalised dict out:
        text, tool_input, citations, search_results, usage, stop_reason."""
        if not self.web_search_ok and self._has_web_search(kwargs):
            kwargs = self._strip_web_search(kwargs)
        if not self.temperature_ok:
            kwargs.pop("temperature", None)
        key = self._key(purpose, tag, kwargs)
        path = self.cache_dir / f"{purpose}_{key}.json"
        if path.exists():
            out = json.loads(path.read_text())
            self._track(purpose, kwargs.get("model"), out.get("usage", {}), True)
            return out
        if self.exhausted:
            raise QuotaExhausted("Free quota already exhausted in this session.")
        try:
            if self.dry_run:
                out = self._fake(purpose, kwargs, key)
            else:
                out = {"anthropic": self._call_anthropic, "gemini": self._call_gemini,
                       "colab_ai": self._call_colab}[self.provider](kwargs)
        except QuotaExhausted:
            self.exhausted = True
            print("\nFREE QUOTA REACHED. Everything done so far is cached. Re-run later (e.g. tomorrow) with "
                  "Runtime > Run all and the notebook continues where it stopped.\n")
            raise
        path.write_text(json.dumps(out))
        self._track(purpose, kwargs.get("model"), out.get("usage", {}), False)
        return out

    # ---------- back-ends ----------
    def _call_anthropic(self, kwargs):
        A = self._anthropic
        messages = list(kwargs["messages"])
        text_parts, citations, results, tool_input = [], [], [], None
        usage = {"input_tokens": 0, "output_tokens": 0, "web_searches": 0}
        resp = None
        for _ in range(6):
            try:
                resp = self.client.messages.create(**{**kwargs, "messages": messages})
            except A.BadRequestError as e:
                msg = str(e).lower()
                if "temperature" in msg and "temperature" in kwargs:
                    self.temperature_ok = False
                    kwargs = {k: v for k, v in kwargs.items() if k != "temperature"}
                    continue
                if self._has_web_search(kwargs):
                    print("Web search tool unavailable, continuing without it:", str(e)[:200])
                    self.web_search_ok = False
                    kwargs = self._strip_web_search(kwargs)
                    continue
                raise
            except A.RateLimitError as e:
                raise QuotaExhausted(str(e)[:300]) if "credit" in str(e).lower() else e
            u = resp.usage
            usage["input_tokens"] += getattr(u, "input_tokens", 0) or 0
            usage["output_tokens"] += getattr(u, "output_tokens", 0) or 0
            stu = getattr(u, "server_tool_use", None)
            if stu is not None:
                usage["web_searches"] += getattr(stu, "web_search_requests", 0) or 0
            for b in resp.content:
                t = getattr(b, "type", None)
                if t == "text":
                    text_parts.append(b.text)
                    for c in (getattr(b, "citations", None) or []):
                        if getattr(c, "url", None):
                            citations.append({"url": c.url, "title": getattr(c, "title", None)})
                elif t == "tool_use":
                    tool_input = b.input
                elif t == "web_search_tool_result":
                    content = getattr(b, "content", None)
                    if isinstance(content, list):
                        for r in content:
                            if getattr(r, "url", None):
                                results.append({"url": r.url, "title": getattr(r, "title", None)})
            if resp.stop_reason == "pause_turn":
                messages = messages + [{"role": "assistant", "content": resp.content}]
                continue
            break
        return {"text": "".join(text_parts).strip(), "tool_input": tool_input, "citations": citations,
                "search_results": results, "usage": usage, "stop_reason": getattr(resp, "stop_reason", None)}

    def _call_gemini(self, kwargs):
        T = self.types
        system, user, tool = self._prompt_parts(kwargs)
        use_search = self._has_web_search(kwargs) and self.web_search_ok

        def build_cfg(search):
            cfg = {"max_output_tokens": max(4096, 4 * int(kwargs.get("max_tokens", 1024)))}  # room for "thinking"
            if "temperature" in kwargs:
                cfg["temperature"] = kwargs["temperature"]
            if system:
                cfg["system_instruction"] = system
            if tool and not search:
                cfg["response_mime_type"] = "application/json"
            if search:
                cfg["tools"] = [T.Tool(google_search=T.GoogleSearch())]
            return T.GenerateContentConfig(**cfg)

        def call(model):
            try:
                return self._retry(lambda: self.client.models.generate_content(
                    model=model, contents=user, config=build_cfg(use_search)))
            except (QuotaExhausted, ModelUnavailable):
                raise
            except Exception as e:
                if use_search and any(s in str(e).lower() for s in ("search", "tool", "grounding")):
                    print("Google Search grounding unavailable, continuing without it:", str(e)[:150])
                    self.web_search_ok = False
                    return self._retry(lambda: self.client.models.generate_content(
                        model=model, contents=user, config=build_cfg(False)))
                raise

        model, resp = self._with_fallback(self._actual(kwargs["model"]), call)
        try:
            text = resp.text or ""
        except Exception:
            text = ""
        cand = (getattr(resp, "candidates", None) or [None])[0]
        gm = getattr(cand, "grounding_metadata", None) if cand is not None else None
        chunks = (getattr(gm, "grounding_chunks", None) or []) if gm is not None else []
        used = set()
        for s in ((getattr(gm, "grounding_supports", None) or []) if gm is not None else []):
            used.update(getattr(s, "grounding_chunk_indices", None) or [])
        cites, results = [], []
        for i, ch in enumerate(chunks):
            web = getattr(ch, "web", None)
            if web is None or not getattr(web, "uri", None):
                continue
            item = {"url": web.uri, "title": getattr(web, "title", None)}
            results.append(item)
            if i in used:
                cites.append(item)
        um = getattr(resp, "usage_metadata", None)
        usage = {"input_tokens": getattr(um, "prompt_token_count", 0) or 0,
                 "output_tokens": (getattr(um, "candidates_token_count", 0) or 0)
                                  + (getattr(um, "thoughts_token_count", 0) or 0),
                 "web_searches": int(bool(chunks))}
        return {"text": text.strip(), "tool_input": self._parse_json(text, tool) if tool else None,
                "citations": cites, "search_results": results, "usage": usage,
                "stop_reason": str(getattr(cand, "finish_reason", "")), "model_used": model}

    def _call_colab(self, kwargs):
        system, user, tool = self._prompt_parts(kwargs)
        prompt = (f"{system}\n\n" if system else "") + user
        def gen(m):
            ex = ThreadPoolExecutor(1)
            fut = ex.submit(self.client.generate_text, prompt, model_name=m)
            try:
                return fut.result(timeout=self.timeout_s)
            except FuturesTimeout:
                raise RuntimeError("timeout: no response from the model")
            finally:
                ex.shutdown(wait=False)
        model, text = self._with_fallback(self._actual(kwargs["model"]), lambda m: self._retry(lambda: gen(m)))
        text = text if isinstance(text, str) else str(text)
        return {"text": text.strip(), "tool_input": self._parse_json(text, tool) if tool else None,
                "citations": [], "search_results": [],
                "usage": {"input_tokens": len(prompt) // 4, "output_tokens": len(text) // 4, "web_searches": 0},
                "stop_reason": "end_turn", "model_used": model}

    def _fake(self, purpose, kwargs, key):
        rng = random.Random(key)
        prompt = kwargs["messages"][-1]["content"]
        prompt = prompt if isinstance(prompt, str) else json.dumps(prompt)
        out = {"text": "", "tool_input": None, "citations": [], "search_results": [],
               "usage": {"input_tokens": len(prompt) // 4, "output_tokens": 150, "web_searches": 0},
               "stop_reason": "end_turn"}
        if purpose in FAKE_HANDLERS:
            out.update(FAKE_HANDLERS[purpose](prompt, kwargs, rng))
        return out

    def usage_report(self):
        df = pd.DataFrame.from_dict(self.usage, orient="index")
        if df.empty:
            return df
        def cost(row):
            p = PRICES_PER_MTOK.get(row["model"])
            if not p:
                return np.nan
            return row["input_tokens"] / 1e6 * p["input"] + row["output_tokens"] / 1e6 * p["output"]
        df["est_cost_usd"] = df.apply(cost, axis=1)
        return df


llm = LLMClient(PROVIDER, api_key=API_KEY, dry_run=DRY_RUN, rpm=RPM_LIMIT)
MODEL_MAIN, MODEL_FAST = llm.resolve_models(MODEL_MAIN, MODEL_FAST)
print(f"Provider: {PROVIDER} | main model: {MODEL_MAIN} | fast model: {MODEL_FAST}")
if PROVIDER == "colab_ai" and USE_WEB_SEARCH:
    print("Note: colab_ai has no web search. The tracker measures what the model recommends from its own "
          "knowledge, and citation analysis is skipped. For grounded answers with citations, use "
          "PROVIDER='gemini' with a free AI Studio key.")
if not DRY_RUN:   # smoke test
    ping = llm.create("smoke_test", model=MODEL_FAST, max_tokens=20,
                      messages=[{"role": "user", "content": "Reply with the single word: ready"}])
    print("Model says:", ping["text"][:80])



Provider: colab_ai | main model: google/gemini-3.8-flash | fast model: google/gemini-3.5-flash-lite
Note: colab_ai has no web search. The tracker measures what the model recommends from its own knowledge, and citation analysis is skipped. For grounded answers with citations, use PROVIDER='gemini' with a free AI Studio key.
Model says: ready


In [6]:
# ---------- Dry-run simulator (only used when DRY_RUN = True) ----------
_FAKE_W = {"Sony": .9, "Bose": .85, "Sennheiser": .45, "Apple": .5, "Bowers & Wilkins": .35, "Anker Soundcore": .45,
           "JBL": .3, "Jabra": .15, "Marshall": .12, "Audio-Technica": .15, "Bang & Olufsen": .1, "Samsung": .12,
           "Nothing": .1, "Beats": .2}
_FAKE_DOMAINS = (DOMAINS["Review / media"][:8] + DOMAINS["Retailer"][:5] + list(DOMAINS["Brand site"])[:6]
                 + DOMAINS["Community / video"][:2])

def _fake_answer(prompt, kwargs, rng):
    w = dict(_FAKE_W)
    if "budget" in prompt or "under £1" in prompt or "value" in prompt:
        w.update({"Anker Soundcore": .9, "JBL": .6, "Sony": .6})
    if "flight" in prompt or "travel" in prompt:
        w.update({"Bose": .95, "Sennheiser": .7})
    chosen = [b for b, p in w.items() if rng.random() < p or b in prompt]
    chosen = sorted(chosen, key=lambda b: -w[b] + rng.random() * 0.6)[:5]
    text = "Here are good options:\n" + "\n".join(f"{i + 1}. **{BRANDS[b][0]}**: a {'top' if i == 0 else 'solid'} choice."
                                                  for i, b in enumerate(chosen))
    urls = [{"url": f"https://www.{d}/best-headphones", "title": d} for d in rng.sample(_FAKE_DOMAINS, 5)]
    if PROVIDER == "colab_ai":   # no web search -> no sources
        urls = []
    return {"text": text, "citations": urls[:3], "search_results": urls,
            "usage": {"input_tokens": 2500, "output_tokens": 350, "web_searches": 1}}

def _fake_extract(prompt, kwargs, rng):
    found = []
    for b, al in BRANDS.items():
        pos = [prompt.find(a) for a in al if a in prompt]
        if pos:
            found.append((min(pos), b))
    recs = [{"brand": b, "product": "", "position": i + 1, "is_top_pick": i == 0,
             "sentiment": rng.choice(["positive", "positive", "neutral"])} for i, (_, b) in enumerate(sorted(found))]
    return {"tool_input": {"recommendations": recs}}

def _fake_memo(prompt, kwargs, rng):
    return {"text": "**Headline:** SIMULATED memo (dry run). Run with a real provider to generate insights.\n\n"
                    "1. Insight placeholder\n2. Insight placeholder\n3. Insight placeholder"}

FAKE_HANDLERS.update({"answer": _fake_answer, "extract": _fake_extract, "memo": _fake_memo})

## 1. Ask the questions
Each question is sent to the AI, and the brands in each answer are saved.

In [7]:
EXTRACT_SCHEMA = {"type": "object", "properties": {"recommendations": {"type": "array", "items": {
    "type": "object", "properties": {
        "brand": {"type": "string"}, "product": {"type": "string"},
        "position": {"type": "integer", "description": "1 = first product mentioned"},
        "is_top_pick": {"type": "boolean", "description": "true only for the overall best recommendation"},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]}},
    "required": ["brand", "product", "position", "is_top_pick", "sentiment"]}}}, "required": ["recommendations"]}

DOMAIN_LIKE = re.compile(r"^[a-z0-9.-]+\.[a-z]{2,}$")


def domain_of(url, title=None):
    """Hostname without www. Gemini grounding returns redirect URLs, and the real domain is in the title."""
    d = urlparse(url).netloc.lower().split(":")[0]
    if ("vertexaisearch" in d or d.endswith("googleusercontent.com")) and title:
        t = title.strip().lower()
        t = t[4:] if t.startswith("www.") else t
        if DOMAIN_LIKE.match(t):
            return t
    return d[4:] if d.startswith("www.") else d


def ask(prompt, rep):
    kw = dict(model=MODEL_MAIN, max_tokens=1500, temperature=1.0, messages=[{"role": "user", "content": prompt}])
    if USE_WEB_SEARCH:
        kw["tools"] = [WEB_SEARCH_TOOL]
    # cache key includes the date: same-day re-runs resume from cache, a new day collects fresh answers
    return llm.create("answer", tag=f"{RUN_TS[:10]}|rep{rep}", **kw)


def extract(answer):
    tool = {"name": "record", "description": "Record the recommendations.", "input_schema": EXTRACT_SCHEMA}
    res = llm.create("extract", model=MODEL_FAST, max_tokens=1500, temperature=0, tools=[tool],
                     tool_choice={"type": "tool", "name": "record"},
                     messages=[{"role": "user", "content": "Extract every product or brand that this AI shopping "
                                f"answer recommends or discusses.\n\n<answer>\n{answer}\n</answer>"}])
    out = []
    for i, x in enumerate((res.get("tool_input") or {}).get("recommendations", []) or []):
        if isinstance(x, dict) and x.get("brand"):
            pos = x.get("position")
            out.append({"brand_raw": str(x["brand"]), "product": str(x.get("product") or ""),
                        "position": int(pos) if isinstance(pos, (int, float)) else i + 1,
                        "is_top_pick": bool(x.get("is_top_pick", False)),
                        "sentiment": x.get("sentiment") if x.get("sentiment") in ("positive", "neutral", "negative") else "neutral"})
    return out


def canonical_brand(name):
    n = (name or "").lower()
    for b, al in BRANDS.items():
        if b.lower() in n or any(a.lower() in n for a in al):
            return b
    return name


# ---------- Preflight: find a model / search combination that actually works on this account ----------
if not DRY_RUN and PROVIDER in ("gemini", "anthropic", "colab_ai"):
    print("Checking which models work on your account (about 30s)…", flush=True)
    cands = list(dict.fromkeys([MODEL_MAIN] + llm.candidates.get("main", []) + [MODEL_FAST] + llm.candidates.get("fast", [])))
    search_wanted = USE_WEB_SEARCH and PROVIDER != "colab_ai"
    chosen, chosen_search = None, False
    for m in cands[:4]:
        if search_wanted:
            ok, err = llm.probe(m, web_search=True)
            print(f"  {m} + web search: {'OK' if ok else 'no (' + err + ')'}", flush=True)
            if ok:
                chosen, chosen_search = m, True
                break
        ok, err = llm.probe(m, web_search=False)
        print(f"  {m}: {'OK' if ok else 'no (' + err + ')'}", flush=True)
        if ok and chosen is None:
            chosen = m
            if not search_wanted:
                break
    if chosen is None:
        raise RuntimeError("No model is answering on this account right now (rate limit or high demand). "
                           "Wait 15-30 minutes and Run all again, or set PROVIDER = 'colab_ai'.")
    if MODEL_MAIN != chosen:
        print(f"Using {chosen} for answers (instead of {MODEL_MAIN}).")
    MODEL_MAIN = chosen
    ok_fast, _ = (True, "") if MODEL_FAST == MODEL_MAIN else llm.probe(MODEL_FAST)
    if not ok_fast:
        print(f"Using {MODEL_MAIN} for extraction too ({MODEL_FAST} is not answering).")
        MODEL_FAST = MODEL_MAIN
    if search_wanted and not chosen_search:
        print("Web search / grounding is not available on this account's free tier, so continuing WITHOUT sources "
              "(the source charts will be empty).")
    USE_WEB_SEARCH = chosen_search
    print(f"Ready: answers={MODEL_MAIN} | extraction={MODEL_FAST} | web search={'on' if USE_WEB_SEARCH else 'off'}", flush=True)

jobs = [(f"P{i:03d}", intent, q + MARKET_SUFFIX, intent == "comparison", r)
        for i, (intent, q) in enumerate((k, q) for k, qs in PROMPTS.items() for q in qs) for r in range(REPEATS)]


_progress = {"done": 0, "t0": time.time()}
_plock = threading.Lock()

def _tick(pid, status):
    with _plock:
        _progress["done"] += 1
        d = _progress["done"]
    print(f"[{d}/{len(jobs)}] {pid} {status} · {time.time() - _progress['t0']:.0f}s elapsed", flush=True)


def work(job):
    pid, intent, prompt, branded, r = job
    row = {"response_id": f"{RUN_ID}-{pid}-{r}", "run_id": RUN_ID, "run_ts": RUN_TS, "provider": PROVIDER,
           "model": MODEL_MAIN, "prompt_id": pid, "intent": intent, "branded": branded, "repeat": r, "prompt": prompt,
           "answer": None, "error": None}
    try:
        res = ask(prompt, r)
        row["answer"], row["model"] = res["text"], res.get("model_used", MODEL_MAIN)
        try:
            recs = extract(res["text"]) if (USE_LLM_EXTRACTION and res["text"]) else []
        except Exception:
            recs = []          # the answer is still usable without structured extraction
        _tick(pid, f"ok ({len(res.get('citations', []))} sources)")
        return row, res.get("citations", []), res.get("search_results", []), recs
    except Exception as e:
        row["error"] = f"{type(e).__name__}: {str(e)[:200]}"
        _tick(pid, f"FAILED: {row['error'][:90]}")
        return row, [], [], []


t0 = time.time()
print(f"Asking {len(jobs)} questions ({MAX_WORKERS} at a time). Expect about {len(jobs) * 2 * 60 / (RPM_LIMIT or 60) / 60:.0f}-"
      f"{len(jobs) * 2 * 60 / (RPM_LIMIT or 60) / 60 * 2:.0f} min on the free tier. Progress is printed below.", flush=True)
with ThreadPoolExecutor(MAX_WORKERS) as ex:
    results = list(ex.map(work, jobs))
raw_responses = pd.DataFrame([r[0] for r in results])
raw_citations = pd.DataFrame([{"response_id": row["response_id"], "run_id": RUN_ID, "source_type": st,
                               "url": c["url"], "title": c.get("title"), "domain": domain_of(c["url"], c.get("title"))}
                              for row, cites, srs, _ in results for st, items in (("citation", cites), ("search_result", srs))
                              for c in items],
                             columns=["response_id", "run_id", "source_type", "url", "title", "domain"])
raw_extractions = pd.DataFrame([{"response_id": row["response_id"], "run_id": RUN_ID, **x,
                                 "brand": canonical_brand(x["brand_raw"])} for row, _, _, recs in results for x in recs],
                               columns=["response_id", "run_id", "brand_raw", "product", "position", "is_top_pick",
                                        "sentiment", "brand"])
n_err = raw_responses["error"].notna().sum()
print(f"Collected {len(raw_responses) - n_err}/{len(raw_responses)} answers, {len(raw_citations)} source links, "
      f"{len(raw_extractions)} extracted recommendations in {time.time() - t0:.0f}s")
if n_err:
    display(raw_responses.loc[raw_responses["error"].notna(), ["prompt_id", "error"]].head())
    if raw_responses["error"].str.contains("QuotaExhausted", na=False).any():
        print("Free quota reached: the analysis below uses the answers collected so far. Re-run later to complete.")
n_ok = int(raw_responses["error"].isna().sum())
if n_ok == 0:
    raise RuntimeError(
        "No answers were collected, so there is nothing to analyse. Most likely the free quota for this provider is "
        "used up for today. Fix: set PROVIDER = 'colab_ai' in the Configuration cell (separate free quota) and choose "
        "Run all again, or retry gemini after 08:00 UK time.")
if n_ok < len(raw_responses) * 0.5:
    print(f"Only {n_ok} answers collected: results will be indicative. Re-run later to complete the set.")
display(raw_responses.loc[raw_responses["answer"].notna(), ["prompt_id", "intent", "prompt", "answer"]].head(3))

Checking which models work on your account (about 30s)…
  google/gemini-3.8-flash: OK
Ready: answers=google/gemini-3.8-flash | extraction=google/gemini-3.5-flash-lite | web search=off
Asking 30 questions (2 at a time). Expect about 4-8 min on the free tier. Progress is printed below.
[1/30] P000 ok (0 sources) · 29s elapsed
[2/30] P001 ok (0 sources) · 36s elapsed
[3/30] P002 ok (0 sources) · 55s elapsed
[4/30] P003 ok (0 sources) · 68s elapsed
[5/30] P004 ok (0 sources) · 85s elapsed
[6/30] P005 ok (0 sources) · 97s elapsed
[7/30] P006 ok (0 sources) · 109s elapsed
[8/30] P007 ok (0 sources) · 125s elapsed
[9/30] P008 ok (0 sources) · 140s elapsed
[10/30] P009 ok (0 sources) · 150s elapsed
[11/30] P010 ok (0 sources) · 165s elapsed
[12/30] P011 ok (0 sources) · 174s elapsed
[13/30] P012 ok (0 sources) · 193s elapsed
[14/30] P013 ok (0 sources) · 202s elapsed
[15/30] P014 ok (0 sources) · 218s elapsed
[16/30] P015 ok (0 sources) · 226s elapsed
[17/30] P016 ok (0 sources) · 242s elapsed

,prompt_id,intent,prompt,answer
0,P000,budget,What are the best noise-cancelling headphones under £100? I'm based in the UK.,"In the UK, the sub-£100 market for active noise-cancelling (ANC) headphones has improved dramatically. While you won..."
1,P001,budget,What are the best noise-cancelling headphones under £150? I'm based in the UK.,"For under £150 in the UK, you are in a sweet spot where you can get genuinely good active noise cancellation (ANC), ..."
2,P002,budget,What are the best noise-cancelling headphones under £200? I'm based in the UK.,"In the UK market, the **under-£200** price point is currently the sweet spot for Active Noise Cancelling (ANC) headp..."


## 2. Save the data and count with SQL

In [8]:
con = duckdb.connect(str(DB_PATH))

def _regex_escape(s):
    return re.sub(r"([.^$*+?()\[\]{}|\\])", r"\\\1", s)

dim_brand = pd.DataFrame({"brand": list(BRANDS), "is_focus": [b == FOCUS_BRAND for b in BRANDS]})
dim_brand_alias = pd.DataFrame([{"brand": b, "alias": a,
                                 "pattern": "(^|[^A-Za-z0-9])" + _regex_escape(a) + "([^A-Za-z0-9]|$)"}
                                for b, al in BRANDS.items() for a in al])
dim_domain = pd.DataFrame(
    [{"domain": d, "category": "Brand site", "owner_brand": o} for d, o in DOMAINS["Brand site"].items()] +
    [{"domain": d, "category": c, "owner_brand": None} for c, ds in DOMAINS.items() if c != "Brand site" for d in ds])

# dimensions are rebuilt each run; raw tables have explicit schemas and are append-only (idempotent per run_id)
for name, df in {"dim_brand": dim_brand, "dim_brand_alias": dim_brand_alias, "dim_domain": dim_domain}.items():
    con.register("_df", df); con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM _df"); con.unregister("_df")

RAW_SCHEMAS = {
    "raw_responses": {"response_id": "VARCHAR", "run_id": "VARCHAR", "run_ts": "VARCHAR", "provider": "VARCHAR",
                      "model": "VARCHAR", "prompt_id": "VARCHAR", "intent": "VARCHAR", "branded": "BOOLEAN",
                      "repeat": "INTEGER", "prompt": "VARCHAR", "answer": "VARCHAR", "error": "VARCHAR"},
    "raw_citations": {"response_id": "VARCHAR", "run_id": "VARCHAR", "source_type": "VARCHAR", "url": "VARCHAR",
                      "title": "VARCHAR", "domain": "VARCHAR"},
    "raw_extractions": {"response_id": "VARCHAR", "run_id": "VARCHAR", "brand_raw": "VARCHAR", "product": "VARCHAR",
                        "position": "INTEGER", "is_top_pick": "BOOLEAN", "sentiment": "VARCHAR", "brand": "VARCHAR"},
}
for name, df in {"raw_responses": raw_responses, "raw_citations": raw_citations,
                 "raw_extractions": raw_extractions}.items():
    cols = RAW_SCHEMAS[name]
    con.execute(f"CREATE TABLE IF NOT EXISTS {name} (" + ", ".join(f'"{c}" {t}' for c, t in cols.items()) + ")")
    con.execute(f"DELETE FROM {name} WHERE run_id = ?", [RUN_ID])
    if len(df):
        con.register("_df", df.astype(object).where(pd.notna(df), None))
        con.execute(f"INSERT INTO {name} SELECT " + ", ".join(f'CAST("{c}" AS {t})' for c, t in cols.items()) + " FROM _df")
        con.unregister("_df")
print({t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("raw_responses", "raw_citations", "raw_extractions")})

{'raw_responses': 30, 'raw_citations': 0, 'raw_extractions': 198}


In [9]:
FOCUS_SQL = FOCUS_BRAND.replace("'", "''")
SQL = {}

SQL["stg_responses"] = """
CREATE OR REPLACE TABLE stg_responses AS
SELECT response_id, run_id,
       CAST(run_ts AS TIMESTAMP)          AS run_ts,
       CAST(CAST(run_ts AS TIMESTAMP) AS DATE) AS run_date,
       provider, model, prompt_id, intent, branded, "repeat", prompt, answer
FROM raw_responses
WHERE error IS NULL AND answer IS NOT NULL AND length(trim(answer)) > 0;
"""

SQL["stg_mentions"] = """
-- brand detected if any alias matches as a whole word (case-sensitive); rank = order of first appearance
CREATE OR REPLACE TABLE stg_mentions AS
WITH hits AS (
    SELECT r.response_id, r.run_id, r.run_date, r.prompt_id, r.intent, r.branded, a.brand,
           MIN(strpos(r.answer, a.alias)) AS first_pos
    FROM stg_responses r
    JOIN dim_brand_alias a ON regexp_matches(r.answer, a.pattern)
    GROUP BY ALL
)
SELECT *, ROW_NUMBER() OVER (PARTITION BY response_id ORDER BY first_pos) AS rank
FROM hits;
"""

SQL["stg_citations"] = """
-- each linked source classified by domain suffix (longest matching rule wins)
CREATE OR REPLACE TABLE stg_citations AS
SELECT c.response_id, c.run_id, c.source_type, c.url, c.domain,
       COALESCE(d.category, 'Other') AS category, d.owner_brand
FROM raw_citations c
JOIN stg_responses r USING (response_id)
LEFT JOIN dim_domain d ON c.domain = d.domain OR c.domain LIKE '%.' || d.domain
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.response_id, c.source_type, c.url
                           ORDER BY length(d.domain) DESC NULLS LAST) = 1;
"""

SQL["mart_share_of_voice"] = """
-- headline KPIs per run and brand (unbranded questions only)
CREATE OR REPLACE TABLE mart_share_of_voice AS
WITH base AS (SELECT * FROM stg_responses WHERE NOT branded),
tot AS (SELECT run_id, run_date, COUNT(*) AS n_answers FROM base GROUP BY ALL),
m AS (SELECT run_id, brand,
             COUNT(DISTINCT response_id)                AS answers_mentioning,
             COUNT(*) FILTER (WHERE rank = 1)           AS times_first,
             AVG(rank)                                  AS avg_rank
      FROM stg_mentions WHERE NOT branded GROUP BY ALL),
allm AS (SELECT run_id, SUM(answers_mentioning) AS total_mentions FROM m GROUP BY run_id)
SELECT t.run_id, t.run_date, b.brand, b.is_focus, t.n_answers,
       COALESCE(m.answers_mentioning, 0)                                     AS answers_mentioning,
       COALESCE(m.answers_mentioning, 0)::DOUBLE / t.n_answers               AS mention_rate,
       COALESCE(m.answers_mentioning, 0)::DOUBLE / NULLIF(a.total_mentions, 0) AS share_of_voice,
       COALESCE(m.times_first, 0)::DOUBLE / t.n_answers                      AS first_position_rate,
       m.avg_rank
FROM tot t
CROSS JOIN dim_brand b
LEFT JOIN m    ON m.run_id = t.run_id AND m.brand = b.brand
LEFT JOIN allm a ON a.run_id = t.run_id
ORDER BY t.run_date, mention_rate DESC;
"""

SQL["mart_sov_by_intent"] = """
CREATE OR REPLACE TABLE mart_sov_by_intent AS
WITH latest AS (SELECT max(run_id) AS run_id FROM stg_responses),
base AS (SELECT * FROM stg_responses WHERE run_id = (SELECT run_id FROM latest) AND NOT branded),
tot AS (SELECT intent, COUNT(*) AS n_answers FROM base GROUP BY intent)
SELECT t.intent, b.brand, b.is_focus, t.n_answers,
       COUNT(DISTINCT m.response_id)::DOUBLE / t.n_answers AS mention_rate
FROM tot t CROSS JOIN dim_brand b
LEFT JOIN stg_mentions m ON m.intent = t.intent AND m.brand = b.brand
                        AND m.run_id = (SELECT run_id FROM latest) AND NOT m.branded
GROUP BY t.intent, b.brand, b.is_focus, t.n_answers
ORDER BY t.intent, mention_rate DESC;
"""

SQL["mart_citation_sources"] = """
-- where the AI's information comes from (cited sources; falls back to retrieved results if nothing is cited)
CREATE OR REPLACE TABLE mart_citation_sources AS
WITH latest AS (SELECT max(run_id) AS run_id FROM stg_responses),
c AS (SELECT DISTINCT response_id, source_type, domain, category, owner_brand
      FROM stg_citations WHERE run_id = (SELECT run_id FROM latest)),
basis AS (SELECT CASE WHEN COUNT(*) FILTER (WHERE source_type = 'citation') > 0
                      THEN 'citation' ELSE 'search_result' END AS b FROM c)
SELECT category, COUNT(*) AS n_links,
       COUNT(*)::DOUBLE / SUM(COUNT(*)) OVER () AS share,
       (SELECT b FROM basis) AS basis
FROM c WHERE source_type = (SELECT b FROM basis)
GROUP BY category ORDER BY n_links DESC;
"""

SQL["mart_top_domains"] = """
CREATE OR REPLACE TABLE mart_top_domains AS
WITH latest AS (SELECT max(run_id) AS run_id FROM stg_responses),
c AS (SELECT DISTINCT response_id, source_type, domain, category, owner_brand
      FROM stg_citations WHERE run_id = (SELECT run_id FROM latest)),
basis AS (SELECT CASE WHEN COUNT(*) FILTER (WHERE source_type = 'citation') > 0
                      THEN 'citation' ELSE 'search_result' END AS b FROM c)
SELECT domain, category, owner_brand, COUNT(DISTINCT response_id) AS answers_citing
FROM c WHERE source_type = (SELECT b FROM basis)
GROUP BY ALL ORDER BY answers_citing DESC, domain LIMIT 20;
"""

SQL["mart_focus_gaps"] = f"""
-- questions where the focus brand is missing, and who wins them instead
CREATE OR REPLACE TABLE mart_focus_gaps AS
WITH latest AS (SELECT max(run_id) AS run_id FROM stg_responses),
r AS (SELECT * FROM stg_responses WHERE run_id = (SELECT run_id FROM latest) AND NOT branded),
focus AS (SELECT DISTINCT response_id FROM stg_mentions WHERE brand = '{FOCUS_SQL}'),
firsts AS (SELECT prompt_id, string_agg(DISTINCT brand, ', ' ORDER BY brand) AS top_picks
           FROM stg_mentions WHERE rank = 1 AND run_id = (SELECT run_id FROM latest) GROUP BY prompt_id)
SELECT r.prompt_id, r.intent, any_value(r.prompt) AS prompt,
       AVG(CASE WHEN f.response_id IS NOT NULL THEN 1.0 ELSE 0.0 END) AS focus_mention_rate,
       any_value(fi.top_picks) AS top_picks
FROM r
LEFT JOIN focus f  ON f.response_id = r.response_id
LEFT JOIN firsts fi ON fi.prompt_id = r.prompt_id
GROUP BY r.prompt_id, r.intent
ORDER BY focus_mention_rate, r.intent, r.prompt_id;
"""

SQL["mart_brand_sentiment"] = """
CREATE OR REPLACE TABLE mart_brand_sentiment AS
WITH latest AS (SELECT max(run_id) AS run_id FROM stg_responses)
SELECT e.brand, COUNT(*) AS recommendations,
       AVG(CASE WHEN e.sentiment = 'positive' THEN 1.0 ELSE 0.0 END) AS positive_share,
       AVG(CASE WHEN e.is_top_pick THEN 1.0 ELSE 0.0 END)            AS top_pick_rate,
       string_agg(DISTINCT NULLIF(e.product, ''), ' | ')             AS products_mentioned
FROM raw_extractions e
JOIN stg_responses r ON r.response_id = e.response_id
WHERE r.run_id = (SELECT run_id FROM latest) AND NOT r.branded
GROUP BY e.brand HAVING COUNT(*) >= 2
ORDER BY recommendations DESC;
"""

for name, sql in SQL.items():
    con.execute(sql)
    (SQL_DIR / f"{name}.sql").write_text(sql.strip() + "\n")
print("Built:", ", ".join(f"{t} ({con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]})" for t in SQL))

Built: stg_responses (30), stg_mentions (118), stg_citations (0), mart_share_of_voice (14), mart_sov_by_intent (56), mart_citation_sources (0), mart_top_domains (0), mart_focus_gaps (26), mart_brand_sentiment (11)


## 3. Results

In [10]:
latest = con.execute("SELECT max(run_id) FROM stg_responses").fetchone()[0]
sov = con.execute("SELECT * FROM mart_share_of_voice WHERE run_id = ?", [latest]).df()
hits = con.execute("""
    SELECT r.prompt_id, r.response_id, b.brand,
           (EXISTS (SELECT 1 FROM stg_mentions m WHERE m.response_id = r.response_id AND m.brand = b.brand))::INT AS hit
    FROM stg_responses r CROSS JOIN dim_brand b
    WHERE r.run_id = ? AND NOT r.branded""", [latest]).df()


def cluster_bootstrap(df, n_boot=2000, seed=42):
    g = df.groupby("prompt_id")["hit"].agg(["sum", "count"])
    if g.empty:
        return np.nan, np.nan
    idx = np.random.default_rng(seed).integers(0, len(g), (n_boot, len(g)))
    rates = g["sum"].values[idx].sum(1) / g["count"].values[idx].sum(1)
    return np.percentile(rates, 2.5), np.percentile(rates, 97.5)


ci = pd.DataFrame([(b, *cluster_bootstrap(h)) for b, h in hits.groupby("brand")], columns=["brand", "ci_low", "ci_high"])
sov = sov.merge(ci, on="brand", how="left").sort_values("mention_rate", ascending=False).reset_index(drop=True)
sov["sov_rank"] = sov["share_of_voice"].rank(ascending=False, method="min").astype(int)
con.register("_df", sov); con.execute("CREATE OR REPLACE TABLE mart_share_of_voice_latest AS SELECT * FROM _df"); con.unregister("_df")

by_intent = con.execute("SELECT * FROM mart_sov_by_intent").df()
sources = con.execute("SELECT * FROM mart_citation_sources").df()
top_domains = con.execute("SELECT * FROM mart_top_domains").df()
gaps = con.execute("SELECT * FROM mart_focus_gaps").df()
sentiment = con.execute("SELECT * FROM mart_brand_sentiment").df()
trend = con.execute("SELECT run_date, brand, AVG(share_of_voice) AS share_of_voice FROM mart_share_of_voice GROUP BY ALL ORDER BY run_date").df()

focus = sov[sov.brand == FOCUS_BRAND].iloc[0]
n_answers = int(focus.n_answers)
n_questions = hits["prompt_id"].nunique()
leader = sov.iloc[0]
prompts_won = int((gaps["top_picks"].fillna("").str.contains(re.escape(FOCUS_BRAND))).sum())
own_site_links = int(top_domains.loc[top_domains.owner_brand == FOCUS_BRAND, "answers_citing"].sum()) if len(top_domains) else 0
KPI = {
    "focus_brand": FOCUS_BRAND, "share_of_voice": float(focus.share_of_voice or 0), "sov_rank": int(focus.sov_rank),
    "n_brands": len(BRANDS), "mention_rate": float(focus.mention_rate),
    "mention_ci": [float(focus.ci_low), float(focus.ci_high)], "first_position_rate": float(focus.first_position_rate),
    "questions_won": prompts_won, "n_questions": n_questions, "n_answers": n_answers,
    "leader": leader.brand, "leader_mention_rate": float(leader.mention_rate),
    "top_source_category": sources.iloc[0]["category"] if len(sources) else None,
    "top_source_share": float(sources.iloc[0]["share"]) if len(sources) else None,
    "focus_own_site_citations": own_site_links,
}
display(sov[["brand", "mention_rate", "ci_low", "ci_high", "share_of_voice", "first_position_rate", "avg_rank", "sov_rank"]]
        .style.format({c: "{:.0%}" for c in ["mention_rate", "ci_low", "ci_high", "share_of_voice", "first_position_rate"]}
                      | {"avg_rank": "{:.1f}"})
        .apply(lambda r: ["background-color:#fff3cd" if r.brand == FOCUS_BRAND else "" for _ in r], axis=1))



,brand,mention_rate,ci_low,ci_high,share_of_voice,first_position_rate,avg_rank,sov_rank
0,Sony,100%,100%,100%,24%,58%,1.7,1
1,Bose,92%,81%,100%,22%,31%,1.9,2
2,Sennheiser,65%,46%,81%,16%,4%,3.1,3
3,Apple,62%,42%,81%,15%,4%,3.8,4
4,Anker Soundcore,46%,27%,65%,11%,0%,3.9,5
5,JBL,15%,4%,31%,4%,0%,3.5,6
6,Jabra,15%,4%,31%,4%,4%,2.0,6
7,Bowers & Wilkins,12%,0%,27%,3%,0%,4.7,8
8,Beats,4%,0%,12%,1%,0%,2.0,9
9,Samsung,4%,0%,12%,1%,0%,3.0,9


In [11]:
FOCUS_C, OTHER_C = "#d6336c", "#adb5bd"
FIGS = {}

plot = sov[sov.mention_rate > 0].copy()
fig = go.Figure(go.Bar(
    x=plot.mention_rate, y=plot.brand, orientation="h",
    marker_color=[FOCUS_C if b == FOCUS_BRAND else OTHER_C for b in plot.brand],
    error_x=dict(type="data", symmetric=False, array=plot.ci_high - plot.mention_rate,
                 arrayminus=plot.mention_rate - plot.ci_low, color="#495057", thickness=1),
    hovertemplate="%{y}: %{x:.0%}<extra></extra>"))
fig.update_layout(title="How often each brand appears in AI answers (95% CI)", xaxis_tickformat=".0%",
                  xaxis_range=[0, min(1.0, float(np.nanmax(plot[["mention_rate", "ci_high"]].values)) * 1.08 + 0.02)],
                  yaxis_autorange="reversed", height=60 + 32 * len(plot), margin=dict(l=10, r=10, t=50, b=10),
                  plot_bgcolor="white")
FIGS["mention_rate"] = fig

pie = sov[sov.share_of_voice > 0].copy()
small = pie.share_of_voice < 0.03
if small.sum() > 1:
    pie = pd.concat([pie[~small], pd.DataFrame([{"brand": "Other", "share_of_voice": pie[small].share_of_voice.sum()}])])
fig = go.Figure(go.Pie(labels=pie.brand, values=pie.share_of_voice, hole=0.55, sort=False,
                       marker_colors=[FOCUS_C if b == FOCUS_BRAND else None for b in pie.brand],
                       textinfo="label+percent", textposition="outside"))
fig.update_layout(title="AI Share of Voice", height=420, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
FIGS["share_of_voice"] = fig

if len(by_intent):
    top_brands = sov[sov.mention_rate > 0].brand.head(8).tolist()
    if FOCUS_BRAND not in top_brands:
        top_brands.append(FOCUS_BRAND)
    heat = by_intent[by_intent.brand.isin(top_brands)].pivot(index="brand", columns="intent", values="mention_rate")
    heat = heat.loc[[b for b in top_brands if b in heat.index]]
    fig = px.imshow(heat, text_auto=".0%", color_continuous_scale="Blues", aspect="auto",
                    title="Who wins which kind of shopper question")
    fig.update_layout(height=80 + 40 * len(heat), coloraxis_showscale=False, margin=dict(l=10, r=10, t=50, b=10))
    FIGS["intent_heatmap"] = fig

if len(sources):
    fig = px.bar(sources, x="share", y="category", orientation="h", text=sources.share.map("{:.0%}".format),
                 title=f"Where the AI's information comes from ({'cited' if sources.basis.iloc[0] == 'citation' else 'retrieved'} sources)",
                 color_discrete_sequence=["#4263eb"])
    fig.update_layout(xaxis_tickformat=".0%", yaxis_autorange="reversed", height=300, plot_bgcolor="white",
                      margin=dict(l=10, r=10, t=50, b=10))
    FIGS["sources"] = fig
    fig = px.bar(top_domains.head(12), x="answers_citing", y="domain", color="category", orientation="h",
                 title="Most-used websites in AI answers")
    fig.update_layout(yaxis_autorange="reversed", height=420, plot_bgcolor="white", margin=dict(l=10, r=10, t=50, b=10))
    FIGS["top_domains"] = fig

if trend.run_date.nunique() > 1:
    t = trend[trend.brand.isin(sov.brand.head(6).tolist() + [FOCUS_BRAND])]
    fig = px.line(t, x="run_date", y="share_of_voice", color="brand", markers=True, title="Share of Voice over time")
    fig.update_layout(yaxis_tickformat=".0%", height=360, plot_bgcolor="white")
    FIGS["trend"] = fig

for f in FIGS.values():
    f.show()

## 4. Check the product page
Can AI bots read the page? Does it have numbers, a spec table, an FAQ and structured data?

In [12]:
# ---------- Page-check engine (shared with GEO Page Doctor) ----------
BROWSER_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/130.0 Safari/537.36")
BOT_UA = "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko); compatible; GPTBot/1.1; +https://openai.com/gptbot"
ROBOTS_AGENTS = {
    "OAI-SearchBot": "search", "ChatGPT-User": "user-fetch", "GPTBot": "training",
    "Claude-SearchBot": "search", "Claude-User": "user-fetch", "ClaudeBot": "training",
    "PerplexityBot": "search", "Perplexity-User": "user-fetch", "Google-Extended": "training", "CCBot": "training",
}


@dataclass
class Fetch:
    status: Optional[int]
    html: str
    error: Optional[str]


def fetch_raw(url, ua=BROWSER_UA, timeout=20):
    headers = {"User-Agent": ua, "Accept-Language": "en-GB,en;q=0.9",
               "Accept": "text/html,application/xhtml+xml,*/*;q=0.8"}
    try:
        with httpx.Client(follow_redirects=True, timeout=timeout, headers=headers) as c:
            r = c.get(url)
        return Fetch(r.status_code, r.text if r.status_code < 400 else "", None)
    except Exception as e:
        return Fetch(None, "", f"{type(e).__name__}: {e}"[:200])


def playwright_ready():
    try:
        import playwright  # noqa: F401
        return True
    except ImportError:
        return False


def fetch_rendered(url, timeout_ms=45000):
    def _run():
        from playwright.sync_api import sync_playwright
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True, args=["--no-sandbox"])
            page = browser.new_page(user_agent=BROWSER_UA, locale="en-GB")
            page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
            try:
                page.wait_for_load_state("networkidle", timeout=15000)
            except Exception:
                pass
            for _ in range(6):
                page.mouse.wheel(0, 2500)
                page.wait_for_timeout(400)
            html = page.content()
            browser.close()
            return html
    try:
        with ThreadPoolExecutor(1) as ex:
            return Fetch(200, ex.submit(_run).result(), None)
    except Exception as e:
        return Fetch(None, "", f"{type(e).__name__}: {e}"[:200])


def check_robots(url):
    p = urlparse(url)
    f = fetch_raw(f"{p.scheme}://{p.netloc}/robots.txt")
    rows = []
    if f.status != 200:
        for a, purpose in ROBOTS_AGENTS.items():
            rows.append({"agent": a, "purpose": purpose, "allowed": True if f.status == 404 else None})
        return pd.DataFrame(rows)
    rp = robotparser.RobotFileParser()
    rp.parse(f.html.splitlines())
    for a, purpose in ROBOTS_AGENTS.items():
        rows.append({"agent": a, "purpose": purpose, "allowed": rp.can_fetch(a, url)})
    return pd.DataFrame(rows)


def _clean_soup(html, drop_layout=False):
    soup = BeautifulSoup(html or "", "lxml")
    junk = ["script", "style", "noscript", "svg", "template", "iframe"]
    if drop_layout:
        junk += ["nav", "header", "footer", "aside", "form"]
    for t in soup(junk):
        t.decompose()
    return soup


def visible_text(html):
    return re.sub(r"\s+", " ", _clean_soup(html).get_text(" ", strip=True))


def main_content_text(html, max_words=900):
    """Best-effort product content: <main>/<article> if present, without nav/header/footer."""
    soup = _clean_soup(html, drop_layout=True)
    cands = soup.find_all(["main", "article"]) + soup.find_all(attrs={"role": "main"})
    node = max(cands, key=lambda n: len(n.get_text(" ", strip=True)), default=None) or soup.body or soup
    text = re.sub(r"\s+", " ", node.get_text(" ", strip=True))
    words = text.split()
    return " ".join(words[:max_words])


WORD_RE = re.compile(r"[a-z0-9][a-z0-9\-\.]{2,}")


def content_coverage(raw_html, rendered_html):
    raw_t, ren_t = visible_text(raw_html), visible_text(rendered_html)
    raw_w, ren_w = set(WORD_RE.findall(raw_t.lower())), set(WORD_RE.findall(ren_t.lower()))
    cov = len(raw_w & ren_w) / len(ren_w) if ren_w else None
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ren_t) if 40 <= len(s.strip()) <= 300]
    hidden = [s for s in sents if s.lower() not in raw_t.lower()]
    return {"coverage": cov, "hidden_sentences": len(hidden), "hidden_examples": hidden[:4],
            "raw_words": len(raw_t.split()), "rendered_words": len(ren_t.split())}

In [13]:
STAT_RE = re.compile(
    r"(?:£|\$|€)\s?\d[\d,]*(?:\.\d+)?"
    r"|\b\d+(?:\.\d+)?\s?(?:%|hours?|hrs?|h\b|mins?|minutes?|mm\b|cm\b|g\b|grams?|kg\b|db\b|hz\b|khz\b|mah\b|"
    r"w\b|ms\b|years?|devices?|mics?|microphones?|reviews?|inch(?:es)?|\"|gb\b|tb\b|mp\b|fps\b|litres?|l\b|ml\b)"
    r"|\bbluetooth\s?\d(?:\.\d)?\b|\b\d(?:\.\d)?\s?/\s?5\b",
    re.I,
)
PRODUCT_FIELDS = ["name", "description", "image", "brand", "sku|gtin|gtin13|gtin12|gtin8|mpn",
                  "offers", "offers.price", "aggregateRating", "review"]


def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(\[£$€])", text or "") if s.strip()]


def text_features(text):
    text = text or ""
    sents = split_sentences(text)
    found = STAT_RE.findall(text)
    lines = [l.strip() for l in re.split(r"[\r\n]+", text)]
    faq = [l for l in lines if l.endswith("?") and 3 <= len(l.split()) <= 25] or \
          [s for s in sents if s.endswith("?") and 3 <= len(s.split()) <= 25]
    return {"word_count": len(text.split()), "unique_stats": len({x.lower().replace(" ", "") for x in found}),
            "quotable_sentences": sum(1 for s in sents if 8 <= len(s.split()) <= 35 and STAT_RE.search(s)),
            "faq_questions": len(faq),
            "table_rows": sum(1 for l in lines if l.startswith("|") and l.count("|") >= 3
                              and not set(l) <= set("|-: "))}


def jsonld_objects(soup):
    objs = []
    for s in soup.find_all("script", attrs={"type": re.compile("ld\\+json", re.I)}):
        try:
            data = json.loads(s.string or s.get_text() or "")
        except Exception:
            continue
        stack = [data]
        while stack:
            d = stack.pop()
            if isinstance(d, list):
                stack.extend(d)
            elif isinstance(d, dict):
                objs.append(d)
                stack.extend(v for v in d.values() if isinstance(v, (list, dict)))
    return objs


def _types(o):
    t = o.get("@type", [])
    return [x.lower() for x in (t if isinstance(t, list) else [t]) if isinstance(x, str)]


def product_schema(objs):
    prods = [o for o in objs if {"product", "productgroup"} & set(_types(o))]
    if not prods:
        return 0.0, {f: False for f in PRODUCT_FIELDS}
    def present(p):
        out = {}
        for f in PRODUCT_FIELDS:
            if f == "offers.price":
                offers = p.get("offers")
                items = offers if isinstance(offers, list) else [offers]
                out[f] = any(isinstance(o, dict) and (o.get("price") or o.get("lowPrice")) for o in items)
            else:
                out[f] = any(p.get(k) for k in f.split("|"))
        return out
    best = max((present(p) for p in prods), key=lambda d: sum(d.values()))
    return sum(best.values()) / len(best), best


def analyze_html(html):
    soup = BeautifulSoup(html or "", "lxml")
    objs = jsonld_objects(soup)
    completeness, fields = product_schema(objs)
    faq_schema = sum((len(o.get("mainEntity")) if isinstance(o.get("mainEntity"), list) else 1)
                     for o in objs if "faqpage" in _types(o))
    spec_pairs = sum(1 for tr in soup.find_all("tr") if len(tr.find_all(["td", "th"])) == 2) + len(soup.find_all("dt"))
    faq_html = sum(1 for t in soup.find_all(["h2", "h3", "h4", "h5", "summary", "dt", "button", "strong"])
                   if t.get_text(strip=True).endswith("?"))
    meta = soup.find("meta", attrs={"name": re.compile("^description$", re.I)})
    html_tag = soup.find("html")
    tf = text_features(visible_text(html))
    return {**tf, "schema_completeness": round(completeness, 2), "schema_fields": fields,
            "microdata_product": bool(soup.find(attrs={"itemtype": re.compile("schema.org/Product", re.I)})),
            "faq_schema": faq_schema, "faq_html": faq_html, "spec_pairs": spec_pairs,
            "h1": len(soup.find_all("h1")), "h2_h3": len(soup.find_all(["h2", "h3"])),
            "title": soup.title.get_text(strip=True) if soup.title else "",
            "meta_description": bool(meta and meta.get("content")), "lang": html_tag.get("lang") if html_tag else None}


def geo_score(f, access=None):
    comp = {}
    if access is not None:
        cov = access.get("coverage")
        comp["AI access"] = 25 * (0.4 * access["robots_allowed_share"] + 0.2 * access["bot_fetch_ok"]
                                  + 0.4 * (1.0 if cov is None else cov))
    comp["Structured data"] = 20 * (f["schema_completeness"] or (0.4 if f["microdata_product"] else 0))
    comp["Facts & specs"] = 10 * min(f["unique_stats"] / 15, 1) + 10 * min(f["spec_pairs"] / 10, 1)
    comp["FAQ"] = 10 * min(max(f["faq_html"], f["faq_schema"], f["faq_questions"]) / 4, 1)
    comp["Structure"] = 3 * (f["h1"] == 1) + 4 * min(f["h2_h3"] / 4, 1) + 3 * min(f["word_count"] / 400, 1)
    comp["Quotability"] = 10 * min(f["quotable_sentences"] / 8, 1)
    comp["Metadata"] = 2 * bool(f["title"]) + 2 * bool(f["meta_description"]) + 1 * bool(f["lang"])
    total = sum(comp.values()) / (100 if access is not None else 75) * 100
    return round(total), {k: round(v, 1) for k, v in comp.items()}


def text_geo_score(text):
    """Content-only score (0–100) for plain text: facts, spec table, FAQ, quotability, length."""
    f = text_features(text)
    comp = {"Facts & specs": 10 * min(f["unique_stats"] / 12, 1) + 10 * min(f["table_rows"] / 8, 1),
            "FAQ": 10 * min(f["faq_questions"] / 4, 1),
            "Quotability": 10 * min(f["quotable_sentences"] / 6, 1),
            "Length": 5 * min(f["word_count"] / 150, 1)}
    return round(sum(comp.values()) / 45 * 100), f


def build_checklist(f, access=None, cov=None, robots=None, bot_status=None):
    """Traffic-light checklist: list of (status, title, detail); status in pass | warn | fail | info."""
    items = []
    add = lambda s, t, d: items.append((s, t, d))
    if access is not None:
        blocked = [] if robots is None else robots.query("allowed == False and purpose != 'training'")["agent"].tolist()
        add("fail" if blocked else "pass", "AI search bots allowed (robots.txt)",
            f"Blocked: {', '.join(blocked)}" if blocked else "Search and user-triggered AI agents may fetch this page.")
        add("pass" if access["bot_fetch_ok"] else "fail", "Page served to AI crawler",
            "An AI-bot user agent received the page." if access["bot_fetch_ok"]
            else f"AI-bot user agent was refused (status {bot_status}). Check CDN/WAF bot rules.")
        if cov is not None and cov.get("coverage") is not None:
            c = cov["coverage"]
            add("pass" if c >= 0.75 else "warn" if c >= 0.4 else "fail", "Content visible without JavaScript",
                f"{c:.0%} of the rendered page's vocabulary is in the raw HTML; "
                f"{cov['hidden_sentences']} sentences appear only after JavaScript runs.")
        else:
            add("info", "Content visible without JavaScript", "Not measured (JavaScript rendering was off).")
        sc = f["schema_completeness"]
        missing = [k.split("|")[0] for k, v in f["schema_fields"].items() if not v]
        add("pass" if sc >= 0.8 else "warn" if sc > 0 else "fail", "Product structured data (JSON-LD)",
            "Complete." if sc >= 0.8 else f"Missing: {', '.join(missing)}." if sc > 0 else "No schema.org Product found.")
    n = f["unique_stats"]
    add("pass" if n >= 8 else "warn" if n >= 3 else "fail", "Concrete facts and numbers",
        f"{n} distinct quantitative facts (battery hours, weight, dimensions, price…). Aim for 8+.")
    sp = f.get("spec_pairs", f.get("table_rows", 0))
    add("pass" if sp >= 6 else "warn" if sp >= 2 else "fail", "Specification table",
        f"{sp} specification rows found. AI engines lift facts from tables easily.")
    q = max(f.get("faq_html", 0), f.get("faq_schema", 0), f.get("faq_questions", 0))
    add("pass" if q >= 3 else "warn" if q >= 1 else "fail", "FAQ / shopper questions answered",
        f"{q} questions found. Answer the questions shoppers actually ask AI assistants.")
    qs = f["quotable_sentences"]
    add("pass" if qs >= 4 else "warn" if qs >= 1 else "fail", "Quotable, fact-bearing sentences",
        f"{qs} short sentences with a number that an AI can quote directly.")
    if access is not None:
        add("pass" if f["meta_description"] and f["title"] else "warn", "Title and meta description",
            "Present." if f["meta_description"] and f["title"] else "Add a factual title and meta description.")
    return items



In [14]:
def page_check(url, render=True):
    fb, fbot = fetch_raw(url), fetch_raw(url, BOT_UA)
    fren = fetch_rendered(url) if (render and playwright_ready()) else None
    rendered = fren.html if fren is not None and not fren.error else None
    if not (fb.html or rendered):
        return {"ok": False, "error": f"Could not fetch the page ({fb.status or fb.error}). Many retailers block "
                                      "automated requests from cloud servers; try the brand's own site."}
    robots = check_robots(url)
    ai_html, full_html = (fbot.html or fb.html), (rendered or fb.html)
    cov = content_coverage(ai_html, rendered) if rendered else None
    relevant = robots[robots.purpose != "training"]["allowed"]
    access = {"robots_allowed_share": float(relevant.fillna(True).astype(bool).mean()),
              "bot_fetch_ok": float(bool(fbot.html)), "coverage": cov["coverage"] if cov else None}
    f_ai = analyze_html(ai_html)
    score, comps = geo_score(f_ai, access)
    potential, _ = geo_score(analyze_html(full_html), {"robots_allowed_share": 1, "bot_fetch_ok": 1, "coverage": 1})
    return {"ok": True, "url": url, "score": score, "potential": potential, "components": comps,
            "checklist": build_checklist(f_ai, access, cov, robots, fbot.status or fbot.error), "coverage": cov}


PAGE = page_check(FOCUS_URL) if FOCUS_URL and not DRY_RUN else None
if DRY_RUN:   # offline demo page (fictional) so the section still renders
    PAGE = {"ok": True, "url": "https://demo.example/product (SIMULATED)", "score": 38, "potential": 74,
            "components": {}, "coverage": {"coverage": 0.21, "hidden_sentences": 7, "hidden_examples": []},
            "checklist": [("fail", "Content visible without JavaScript", "21% of the rendered page's vocabulary is in the raw HTML."),
                          ("warn", "Product structured data (JSON-LD)", "Missing: description, sku, aggregateRating."),
                          ("fail", "FAQ / shopper questions answered", "0 questions found.")]}
if PAGE and PAGE["ok"]:
    print(f"GEO readiness of {PAGE['url']}: {PAGE['score']}/100 as AI crawlers see it "
          f"(potential {PAGE['potential']}/100 if fully visible)")
    display(pd.DataFrame(PAGE["checklist"], columns=["status", "check", "detail"]))
elif PAGE:
    print(PAGE["error"])

GEO readiness of https://uk.sennheiser-hearing.com/products/momentum-4-wireless: 73/100 as AI crawlers see it (potential 83/100 if fully visible)


,status,check,detail
0,pass,AI search bots allowed (robots.txt),Search and user-triggered AI agents may fetch this page.
1,pass,Page served to AI crawler,An AI-bot user agent received the page.
2,warn,Content visible without JavaScript,56% of the rendered page's vocabulary is in the raw HTML; 64 sentences appear only after JavaScript runs.
3,warn,Product structured data (JSON-LD),"Missing: brand, aggregateRating, review."
4,pass,Concrete facts and numbers,"36 distinct quantitative facts (battery hours, weight, dimensions, price…). Aim for 8+."
5,fail,Specification table,0 specification rows found. AI engines lift facts from tables easily.
6,pass,FAQ / shopper questions answered,3 questions found. Answer the questions shoppers actually ask AI assistants.
7,pass,"Quotable, fact-bearing sentences",6 short sentences with a number that an AI can quote directly.
8,pass,Title and meta description,Present.


## 5. Short summary
The AI writes a short summary from the numbers.

In [15]:
def pct(x):
    return None if x is None or (isinstance(x, float) and np.isnan(x)) else round(float(x) * 100)

findings = {
    "category": CATEGORY, "market": "UK", "run_date": RUN_TS[:10], "provider": PROVIDER,
    "sample": {"questions": n_questions, "answers": n_answers, "repeats": REPEATS},
    "focus": {**{k: (pct(v) if k in ("share_of_voice", "mention_rate", "first_position_rate", "leader_mention_rate",
                                     "top_source_share") else v) for k, v in KPI.items()},
              "mention_ci_pct": [pct(KPI["mention_ci"][0]), pct(KPI["mention_ci"][1])]},
    "brands": [{"brand": r.brand, "mention_rate_pct": pct(r.mention_rate), "sov_pct": pct(r.share_of_voice),
                "first_pct": pct(r.first_position_rate)} for r in sov.head(8).itertuples()],
    "focus_by_intent_pct": {r.intent: pct(r.mention_rate) for r in by_intent[by_intent.brand == FOCUS_BRAND].itertuples()},
    "intent_leaders": {i: g.sort_values("mention_rate", ascending=False).brand.iloc[0] for i, g in by_intent.groupby("intent")} if len(by_intent) else {},
    "weakest_questions": gaps.head(5)[["prompt", "focus_mention_rate", "top_picks"]].to_dict("records"),
    "source_mix_pct": {r.category: pct(r.share) for r in sources.itertuples()} if len(sources) else "not available (no web search)",
    "top_domains": top_domains.head(8)[["domain", "category", "answers_citing"]].to_dict("records") if len(top_domains) else [],
    "page_check": {k: PAGE[k] for k in ("score", "potential")} | {"failed_checks": [c[1] for c in PAGE["checklist"] if c[0] == "fail"]}
                  if PAGE and PAGE.get("ok") else None,
}
MEMO_PROMPT = (f"Write a concise memo (max 350 words, Markdown) to the Marketing Director of {FOCUS_BRAND}: "
               "'AI Share of Voice: where we stand and what to do'. Sections: **Headline** (one sentence with the key "
               "number), **3 insights** (each with a number from the data), **3 recommended actions** (specific, "
               "prioritised, each linked to an insight), **Caveat** (one line on sample size/method). Use ONLY these "
               "numbers; do not invent any. Data:\n\n" + json.dumps(findings, default=str, indent=1))
try:
    MEMO = llm.create("memo", model=MODEL_MAIN, max_tokens=1500, temperature=0.2,
                      system="You are a senior marketing analyst. Be specific, quantitative and commercially minded.",
                      messages=[{"role": "user", "content": MEMO_PROMPT}])["text"]
except Exception as e:
    MEMO = (f"**Headline:** {FOCUS_BRAND} appears in {pct(KPI['mention_rate'])}% of AI answers "
            f"(rank {KPI['sov_rank']} of {KPI['n_brands']}); {KPI['leader']} leads with {pct(KPI['leader_mention_rate'])}%.\n\n"
            f"*(Auto-generated summary: the memo model was unavailable: {type(e).__name__}.)*")
display(Markdown(MEMO))

### Headline
Sennheiser holds a 16% AI Share of Voice in the UK noise-cancelling category (rank 3 of 14), but faces a critical conversion bottleneck with a first-position recommendation rate of just 4%.

---

### 3 Insights

1. **Conversion Gap at the Top**: Despite a robust 65% mention rate (CI: 46%–81%), Sennheiser converts only 4% into first-position recommendations (1 won out of 26 questions), trailing Sony (58%) and Bose (31%).
2. **Collapse in Use-Case Relevance**: While Sennheiser captures 100% of generic and 80% of budget intents, performance drops to 25% for use-case queries—plunging to a 0.0% mention rate for London Underground commuting and working-from-home calls.
3. **Owned Technical Content Deficit**: Sennheiser secured 0 own-site citations and scored 73 out of an 83 potential on technical page readiness, directly failing the "Specification table" check.

---

### 3 Recommended Actions

1. **Priority 1: Reclaim High-Value Commuter & WFH Intents (Linked to Insight 2)**  
   Produce contextual campaigns addressing specific UK scenarios—specifically targeting London Underground noise cancellation and call-microphone clarity—to lift use-case mention rates from 25% where Sony and Bose currently dominate.
2. **Priority 2: Optimise Positioning for First-Rank Conversion (Linked to Insight 1)**  
   Re-engineer product messaging to directly contest premium attributes (where Sennheiser currently scored 0.0% for "best noise cancellation" and "best mic"), closing the gap between the 65% mention rate and the 4% top-recommendation rate.
3. **Priority 3: Remediate On-Site Technical Schema (Linked to Insight 3)**  
   Deploy standard specification tables across UK product landing pages to capture the unrealised 10-point quality score gain (moving from 73 to 83) and establish indexable structured data to lift own-site citations above 0.

---

### Caveat
Methodology is limited to a small sample of 26 prompts run without live web retrieval via colab_ai on 2026-11-25.

## 6. Save the dashboard and files

In [16]:
import markdown as mdlib

def _e(x):
    return htmllib.escape(str(x))

def tile(label, value, sub=""):
    return f'<div class="tile"><div class="tl">{_e(label)}</div><div class="tv">{value}</div><div class="ts">{_e(sub)}</div></div>'

ci_txt = f"95% CI {pct(KPI['mention_ci'][0])}–{pct(KPI['mention_ci'][1])}%"
tiles = "".join([
    tile("AI Share of Voice", f"{pct(KPI['share_of_voice'])}%", f"rank {KPI['sov_rank']} of {KPI['n_brands']} brands"),
    tile("Mentioned in", f"{pct(KPI['mention_rate'])}%", f"of AI answers · {ci_txt}"),
    tile("Recommended first", f"{pct(KPI['first_position_rate'])}%", f"leader: {KPI['leader']} ({pct(KPI['leader_mention_rate'])}% mentioned)"),
    tile("Questions won", f"{KPI['questions_won']}/{KPI['n_questions']}", "focus brand named first"),
    tile("Top source type", f"{pct(KPI['top_source_share'])}%" if KPI["top_source_share"] is not None else "n/a",
         KPI["top_source_category"] or "needs web search"),
] + ([tile("Page GEO score", f"{PAGE['score']}/100", f"potential {PAGE['potential']}/100")] if PAGE and PAGE.get("ok") else []))

gap_rows = "".join(f"<tr><td>{_e(r.intent)}</td><td>{_e(r.prompt)}</td><td>{pct(r.focus_mention_rate)}%</td>"
                   f"<td>{_e(r.top_picks or '–')}</td></tr>" for r in gaps.head(8).itertuples())
page_rows = "".join(f"<tr><td>{'✅' if s == 'pass' else '⚠️' if s == 'warn' else '❌' if s == 'fail' else 'ℹ️'}</td>"
                    f"<td>{_e(t)}</td><td>{_e(d)}</td></tr>" for s, t, d in (PAGE["checklist"] if PAGE and PAGE.get("ok") else []))
figs_html = "".join(f'<div class="card">{f.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False)}</div>'
                    for i, f in enumerate(FIGS.values()))
banner = '<div class="banner">SIMULATED DATA (dry run). Not real AI answers.</div>' if DRY_RUN else ""

DASHBOARD = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<title>GEO Project: AI Share of Voice · {_e(CATEGORY)} · {RUN_TS[:10]}</title>
<style>body{{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;background:#f8f9fa;color:#212529;margin:0}}
.wrap{{max-width:1150px;margin:0 auto;padding:28px 20px}} h1{{margin:0;font-size:1.7rem}} .sub{{color:#6c757d}}
.tiles{{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin:18px 0}}
.tile{{background:#fff;border-radius:14px;padding:14px 16px;box-shadow:0 1px 3px rgba(0,0,0,.08)}}
.tl{{font-size:.8rem;color:#6c757d}} .tv{{font-size:2rem;font-weight:700;color:{FOCUS_C}}} .ts{{font-size:.78rem;color:#868e96}}
.grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(480px,1fr));gap:14px}}
.card{{background:#fff;border-radius:14px;padding:10px;box-shadow:0 1px 3px rgba(0,0,0,.08);margin-bottom:14px;overflow:auto}}
.memo{{background:#fff;border-left:5px solid {FOCUS_C};border-radius:10px;padding:6px 22px;margin:14px 0;box-shadow:0 1px 3px rgba(0,0,0,.08)}}
table{{border-collapse:collapse;width:100%;font-size:.88rem}} td,th{{padding:6px 8px;border-bottom:1px solid #eee;text-align:left}}
.banner{{background:#ffe3e3;border:1px solid #ffa8a8;padding:8px 12px;border-radius:8px;font-weight:600;margin-bottom:10px}}
h2{{font-size:1.15rem;margin:22px 0 8px}} .foot{{font-size:.8rem;color:#868e96;margin-top:20px}}</style></head>
<body><div class="wrap">{banner}
<h1>GEO Project: AI Share of Voice</h1>
<div class="sub">{_e(CATEGORY)} · UK · focus brand <b>{_e(FOCUS_BRAND)}</b> · {n_questions} shopper questions, {n_answers} AI answers ·
model provider {_e(PROVIDER)} · {RUN_TS}</div>
<div class="tiles">{tiles}</div>
<div class="memo">{mdlib.markdown(MEMO)}</div>
<div class="grid">{figs_html}</div>
<h2>Opportunity gaps: questions where {_e(FOCUS_BRAND)} is missing</h2>
<div class="card"><table><tr><th>Intent</th><th>Shopper question</th><th>{_e(FOCUS_BRAND)} mentioned</th><th>Recommended first instead</th></tr>{gap_rows}</table></div>
{f'<h2>Product page GEO readiness: {_e(PAGE["url"])}</h2><div class="card"><table>{page_rows}</table></div>' if page_rows else ''}
<div class="foot">Method: each question was sent to an AI assistant {"with web search/grounding " if USE_WEB_SEARCH and PROVIDER != "colab_ai" else ""}
and no system prompt; brands detected by whole-word alias matching in SQL; mention-rate CIs by cluster bootstrap over
questions; comparison questions that name brands are excluded from KPIs. Independent analysis, not affiliated with any brand.</div>
</div></body></html>"""
dash_path = BASE_DIR / ("dashboard_SIMULATED.html" if DRY_RUN else f"dashboard_{RUN_ID}.html")
dash_path.write_text(DASHBOARD, encoding="utf-8")
(BASE_DIR / "memo.md").write_text(MEMO, encoding="utf-8")
(BASE_DIR / "findings.json").write_text(json.dumps(findings, default=str, indent=2))

EXPORT_TABLES = ["mart_share_of_voice", "mart_share_of_voice_latest", "mart_sov_by_intent", "mart_citation_sources",
                 "mart_top_domains", "mart_focus_gaps", "mart_brand_sentiment", "stg_mentions", "stg_responses"]
(BASE_DIR / "csv").mkdir(exist_ok=True)
for t in EXPORT_TABLES:
    con.execute(f"SELECT * FROM {t}").df().to_csv(BASE_DIR / "csv" / f"{t}.csv", index=False)
print("Dashboard:", dash_path.resolve())
print("CSV + SQL + DuckDB:", BASE_DIR.resolve())

Dashboard: /content/drive/MyDrive/geo-project/dashboard_20261125-144739.html
CSV + SQL + DuckDB: /content/drive/MyDrive/geo-project


In [17]:
# ---------- Google Sheets (data source for Looker Studio) ----------
SHEET_URL = None
if IN_COLAB and EXPORT_TO_SHEETS and not DRY_RUN:
    try:
        from google.colab import auth
        import gspread
        from google.auth import default
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        title = f"GEO Project: {CATEGORY} (data)"
        try:
            sh = gc.open(title)
        except gspread.SpreadsheetNotFound:
            sh = gc.create(title)
        existing = {ws.title: ws for ws in sh.worksheets()}
        for t in EXPORT_TABLES:
            df = con.execute(f"SELECT * FROM {t}").df()
            if "answer" in df:
                df["answer"] = df["answer"].str.slice(0, 4000)
            values = [df.columns.tolist()] + json.loads(df.to_json(orient="values", date_format="iso"))
            ws = existing.get(t) or sh.add_worksheet(title=t, rows=max(len(values) + 10, 50), cols=max(len(df.columns), 5))
            ws.clear()
            ws.resize(rows=max(len(values) + 10, 50), cols=max(len(df.columns), 5))
            ws.update(range_name="A1", values=values)
        if "Sheet1" in existing and len(existing) > 1:
            sh.del_worksheet(existing["Sheet1"])
        SHEET_URL = sh.url
        print("Google Sheet for Looker Studio:", SHEET_URL)
    except Exception as e:
        print("Google Sheets export skipped:", type(e).__name__, str(e)[:150])

display(HTML(DASHBOARD))
print("\nUsage:"); display(llm.usage_report())

Google Sheet for Looker Studio: https://docs.google.com/spreadsheets/d/177DvM7nk1jzn7FrOyy8yJJkjvKP5K3rCth9n0Jld-0o



Usage:


,model,calls,cache_hits,input_tokens,output_tokens,web_searches,est_cost_usd
smoke_test,google/gemini-3.5-flash-lite,1,0,8,1,0,NaN
answer,google/gemini-3.8-flash,30,0,660,29893,0,NaN
extract,google/gemini-3.5-flash-lite,30,0,35457,5677,0,NaN
memo,google/gemini-3.8-flash,1,0,810,499,0,NaN
